In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import numpy as np
import classical_algo

FILEPATH = "/work/Osaba_reimplementation/Osaba_paper_data/data/d4_P0.pdb"
vehicles, nodes, distances = classical_algo.readInData(FILEPATH)

In [ ]:
def plot_routes(vehicles, nodes, routes_per_vehicle):
    """
    vehicles          : list of Vehicle objects
    nodes             : list of Node objects (index == node_id)
    routes_per_vehicle: dict  {vehicle_index: [list of node_ids in order]}
                        Build this as you run your main loop (see cell below).
    """
    fig, ax = plt.subplots(figsize=(12, 9))
    ax.set_facecolor('#0d0f14')
    fig.patch.set_facecolor('#0d0f14')

    colors = cm.tab10(np.linspace(0, 1, max(len(routes_per_vehicle), 1)))

    depot = nodes[0]

    # --- draw routes ---
    for idx, (v_idx, route) in enumerate(routes_per_vehicle.items()):
        color = colors[idx % len(colors)]
        full_route = [depot.node_id] + route + [depot.node_id]

        xs = [nodes[n].x for n in full_route]
        ys = [nodes[n].y for n in full_route]

        ax.plot(xs, ys, '-', color=color, linewidth=1.4,
                alpha=0.7, zorder=2)

        # direction arrows at midpoints
        for i in range(len(full_route) - 1):
            mx = (nodes[full_route[i]].x + nodes[full_route[i+1]].x) / 2
            my = (nodes[full_route[i]].y + nodes[full_route[i+1]].y) / 2
            dx = nodes[full_route[i+1]].x - nodes[full_route[i]].x
            dy = nodes[full_route[i+1]].y - nodes[full_route[i]].y
            ax.annotate('', xy=(mx + dx*0.01, my + dy*0.01),
                        xytext=(mx - dx*0.01, my - dy*0.01),
                        arrowprops=dict(arrowstyle='->', color=color,
                                        lw=1.2, alpha=0.8),
                        zorder=3)

    # --- draw nodes ---
    for node in nodes:
        if node.node_id == depot.node_id:
            continue
        style = dict(marker='o', markersize=6, zorder=4, markeredgewidth=0.5)
        if node.is_tp:
            ax.plot(node.x, node.y, color='#ffcc44',
                    markeredgecolor='#ffaa00', **style)
        else:
            delivered_color = '#44ff99' if node.delivered else '#ff4466'
            ax.plot(node.x, node.y, color=delivered_color,
                    markeredgecolor='white', **style)

        ax.annotate(str(node.node_id),
                    xy=(node.x, node.y), xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=6.5, color='#aabbcc', zorder=5)

    # --- depot ---
    ax.plot(depot.x, depot.y, marker='*', markersize=16,
            color='white', markeredgecolor='#5a7aff',
            markeredgewidth=1.2, zorder=6)
    ax.annotate('DEPOT', xy=(depot.x, depot.y), xytext=(6, 6),
                textcoords='offset points',
                fontsize=8, color='white', fontweight='bold', zorder=7)

    # --- legend ---
    vehicle_patches = [
        mpatches.Patch(color=colors[i % len(colors)], label=f'Vehicle {v_idx} ({len(r)} stops)')
        for i, (v_idx, r) in enumerate(routes_per_vehicle.items())
    ]
    status_patches = [
        mpatches.Patch(color='#44ff99', label='Delivered'),
        mpatches.Patch(color='#ff4466', label='Undelivered'),
        mpatches.Patch(color='#ffcc44', label='Time-priority (TP)'),
    ]
    legend1 = ax.legend(handles=vehicle_patches, loc='upper left',
                        fontsize=8, framealpha=0.2,
                        labelcolor='white', facecolor='#1a1f2e')
    ax.add_artist(legend1)
    ax.legend(handles=status_patches, loc='lower left', fontsize=8, framealpha=0.2, labelcolor='white', facecolor='#1a1f2e')

    ax.set_title('Delivery Routes', color='white', fontsize=13, pad=12)
    ax.tick_params(colors='#3d4560')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e2330')
    ax.grid(True, color='#1e2330', linewidth=0.5, linestyle='--')

    plt.tight_layout()
    plt.show()

: 

In [ ]:
# ---------------------------------------------------------------
# Re-run your main loop and collect routes as you go
# ---------------------------------------------------------------
import dimod
import quantum_code_Dwave_copy  as quantum_algo # your find_route module — rename if needed

vehicles, nodes, distances = classical_algo.readInData(FILEPATH)

owned  = sorted([v for v in vehicles if v.rental_cost == 0],
                key=lambda v: v.remaining_weight * v.remaining_volume)
rental = sorted([v for v in vehicles if v.rental_cost > 0],
                key=lambda v: v.remaining_weight * v.remaining_volume)
vehicleStack = owned + rental
tpQueue = sorted([n for n in nodes if n.is_tp], key=lambda n: n.deadline)

routes_per_vehicle = {}   # {vehicle_index: [node_ids]}

while vehicleStack and not classical_algo.all_delivered(nodes):
    nextVehicle = vehicleStack.pop()
    v_idx = id(nextVehicle)   # unique key per vehicle object

    nextDest = classical_algo.pickNextRoute(nextVehicle, tpQueue, distances)
    print(f"Next destination: node {nextDest.node_id}")
    route, route_duration = quantum_algo.find_route(
        nextVehicle, nextDest, nodes, distances)
    print(f"Route returned: {[n.node_id for n in route]}")
    route_ids = [n.node_id for n in route]
    routes_per_vehicle.setdefault(v_idx, []).extend(route_ids)
    for node in route:
        nodes[node.node_id].delivered = True
        nextVehicle.remaining_weight -= node.package_weight
        nextVehicle.remaining_volume -= node.package_volume
        if node.is_tp:
            try:
                tpQueue.remove(node)
            except ValueError:
                pass
    
    nextVehicle.time_spent += route_duration
    nextVehicle.current_location = nextDest.node_id
    if nextDest.node_id != classical_algo.DEPOT.node_id:
        vehicleStack.insert(0, nextVehicle)

plot_routes(vehicles, nodes, routes_per_vehicle)

Next destination: node 1
